# IMO Health — Diagnosis Inference Agent (Medication-Guided)

This notebook provides a **Diagnosis Inference Agent** that infers the most specific diagnosis
from a base diagnosis plus the patient's medications, by traversing the IMO Knowledge Graph
via `domainNarrowerByMedications`.

## What Does This Agent Do?

Given a clinical note, this agent will:
1. **Extract** the base diagnosis and current medications from the note
2. **Normalize** the diagnosis (domain=Problem) and each medication (domain=Medication) via IMO Precision Normalize API
3. **Iteratively drill down** the problem hierarchy using the KG `domainNarrowerByMedications` field
4. **Select the best candidate** at each level using clinical context from the note
5. **Present** the inferred most-specific diagnosis with the full refinement path

| Component | Technology |
|-----------|------------|
| LLM | AWS Bedrock / OpenAI / Anthropic / Azure OpenAI (configurable) |
| Tools | IMO Normalize API + KG GraphQL (`domainNarrowerByMedications`) |
| Agent | LangGraph ReAct Agent |
| Auth | OAuth2 client_credentials grant |

## Prerequisites

- `config.json` file with IMO API + LLM credentials (copy from `config.json.template`)
- For AWS Bedrock: AWS credentials (SageMaker execution role or explicit keys)
- For OpenAI/Anthropic/Azure: respective API keys in config.json
- Python 3.10+

## Architecture Flow

![Architecture Flow](architecture_flow.png)

## Step 1: Install Dependencies

Run this cell once, then restart the kernel.

Installs support for all LLM providers. You only need credentials for the one you choose.

In [ ]:
%pip install -q --upgrade \
    "langchain-core>=1.0.0" \
    "langchain>=1.0.0" \
    "langchain-aws>=0.2.0" \
    "langchain-openai>=0.3.0" \
    "langchain-anthropic>=0.3.0" \
    "langgraph>=0.2.0" \
    boto3 botocore requests nest_asyncio python-dotenv

## Step 2: Configuration

Loads credentials from `config.json`. Create one from `config.json.template` if it doesn't exist.

### LLM Provider

Set `llm.provider` in `config.json` to one of:
- `"bedrock"` — AWS Bedrock (Claude via AWS)
- `"openai"` — OpenAI (GPT-4o, etc.)
- `"anthropic"` — Anthropic direct API (Claude)
- `"azure_openai"` — Azure OpenAI Service

In [ ]:
import os
import sys
import json
import pathlib
import nest_asyncio

nest_asyncio.apply()

# --- Ensure notebook directory is on path for local imports ---
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

# --- Load config.json ---
candidates = [
    pathlib.Path('config.json'),
    pathlib.Path.cwd() / 'config.json',
    pathlib.Path.cwd().parent / 'config.json',
]
cfg_path = next((p for p in candidates if p.exists()), None)
if cfg_path is None:
    raise FileNotFoundError(
        'config.json not found. Copy config.json.template to config.json and fill in your credentials.'
    )

with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

# --- IMO API credentials ---
imo_cfg = cfg.get('imo_api', {})
imo_kg_cfg = cfg.get('imo_kg', {})

os.environ['IMO_NORMALIZE_CLIENT_ID'] = imo_cfg.get('client_id', '')
os.environ['IMO_NORMALIZE_SECRET'] = imo_cfg.get('client_secret', '')
os.environ['IMO_KG_CLIENT_ID'] = imo_kg_cfg.get('client_id', '') or imo_cfg.get('client_id', '')
os.environ['IMO_KG_CLIENT_SECRET'] = imo_kg_cfg.get('client_secret', '') or imo_cfg.get('client_secret', '')

# --- LLM configuration ---
llm_cfg = cfg.get('llm', {})
LLM_PROVIDER = llm_cfg.get('provider', 'bedrock')

# --- AWS credentials from config.json (for Bedrock) ---
aws_cfg = cfg.get('aws', {})
if aws_cfg.get('access_key_id'):
    os.environ.pop('AWS_PROFILE', None)
    os.environ['AWS_ACCESS_KEY_ID'] = aws_cfg['access_key_id']
    os.environ['AWS_SECRET_ACCESS_KEY'] = aws_cfg['secret_access_key']
    os.environ['AWS_SESSION_TOKEN'] = aws_cfg.get('session_token', '')
    os.environ['AWS_DEFAULT_REGION'] = aws_cfg.get('region', 'us-east-1')

print(f'Config loaded from : {cfg_path.resolve()}')
print(f'LLM Provider       : {LLM_PROVIDER}')
print(f'Normalize URL      : {imo_cfg.get("normalize_url", "https://api.imohealth.com/precision/normalize")}')
print(f'KG GraphQL URL     : {imo_cfg.get("graphql_url", "https://api.imohealth.com/knowledgegraph/graphql/")}')

## Step 3: Initialize LLM

Creates the LLM client based on the configured provider.  
Temperature is set to 0 for deterministic outputs across all providers.

In [ ]:
def create_llm(provider: str, llm_cfg: dict):
    """Create LLM instance based on the configured provider."""

    if provider == 'bedrock':
        from langchain_aws import ChatBedrockConverse
        bedrock_cfg = llm_cfg.get('bedrock', {})
        model_id = bedrock_cfg.get('model_id', 'us.anthropic.claude-haiku-4-5-20251001-v1:0')
        region = bedrock_cfg.get('region', 'us-east-1')
        llm = ChatBedrockConverse(
            model_id=model_id,
            region_name=region,
            temperature=0,
            max_tokens=64000,
            provider="anthropic",
        )
        print(f'LLM ready: AWS Bedrock ({model_id}, {region})')
        return llm

    elif provider == 'openai':
        from langchain_openai import ChatOpenAI
        openai_cfg = llm_cfg.get('openai', {})
        model = openai_cfg.get('model', 'gpt-4o')
        llm = ChatOpenAI(
            model=model,
            api_key=openai_cfg.get('api_key', ''),
            temperature=0,
        )
        print(f'LLM ready: OpenAI ({model})')
        return llm

    elif provider == 'anthropic':
        from langchain_anthropic import ChatAnthropic
        anthropic_cfg = llm_cfg.get('anthropic', {})
        model = anthropic_cfg.get('model', 'claude-sonnet-4-20250514')
        llm = ChatAnthropic(
            model=model,
            api_key=anthropic_cfg.get('api_key', ''),
            temperature=0,
            max_tokens=64000,
        )
        print(f'LLM ready: Anthropic ({model})')
        return llm

    elif provider == 'azure_openai':
        from langchain_openai import AzureChatOpenAI
        azure_cfg = llm_cfg.get('azure_openai', {})
        llm = AzureChatOpenAI(
            azure_deployment=azure_cfg.get('deployment', ''),
            azure_endpoint=azure_cfg.get('endpoint', ''),
            api_key=azure_cfg.get('api_key', ''),
            api_version=azure_cfg.get('api_version', '2024-02-15-preview'),
            temperature=0,
        )
        print(f'LLM ready: Azure OpenAI ({azure_cfg.get("deployment", "")})')
        return llm

    else:
        raise ValueError(
            f'Unknown LLM provider: "{provider}". '
            f'Supported: bedrock, openai, anthropic, azure_openai'
        )


llm = create_llm(LLM_PROVIDER, llm_cfg)

## Step 4: Define Agent Tools

Two tools power the dx-infer workflow:

| Tool | Purpose |
|------|---------|
| `normalize_medical_term` | Normalize diagnosis (domain=Problem) or medications (domain=Medication) via IMO Precision Normalize API |
| `get_medication_diagnosis_proto` | Query KG `domainNarrowerByMedications` — returns narrower problem concepts linked to supplied medications |

In [ ]:
from langchain_core.tools import tool
from kg_api_client import KGApiClient

_kg_client = KGApiClient()


@tool
def normalize_medical_term(input_term: str, domain: str = "Problem") -> dict:
    """Normalize a medical term using IMO Precision Normalize API.

    Use domain="Problem" for diagnoses/conditions.
    Use domain="Medication" for medications/drugs.

    Returns the normalized concept with:
    - title: canonical name
    - default_lexical_code: the stable IMO identifier used for KG lookups
    - score: confidence score
    - icd10_codes: mapped ICD-10-CM codes (for Problem domain)
    """
    return _kg_client.normalize_medical_term(input_term, domain)


@tool
def get_medication_diagnosis_proto(diagnosis_code: str, medication_codes: list[str]) -> dict:
    """Query the IMO Knowledge Graph for narrower diagnosis concepts linked to medications.

    Uses the `domainNarrowerByMedications` field on ProblemLexical to find
    child concepts of the base diagnosis that have therapeutic ties to the
    supplied medications.

    Args:
        diagnosis_code: The base diagnosis lexical code (from normalize with domain="Problem").
        medication_codes: List of medication lexical codes (from normalize with domain="Medication").

    Returns:
        Dict with the base diagnosis title and its domainNarrowerByMedications list.
        Each entry has: code, title, numberOfDomainChildren.
        - numberOfDomainChildren > 0 means the concept can be drilled further
        - numberOfDomainChildren == 0 means it is a leaf (most specific)

    Use this tool iteratively: pick the best candidate matching clinical context,
    then call again with that candidate's code to drill deeper, until a leaf is reached.
    """
    return _kg_client.get_medication_diagnosis_proto(diagnosis_code, medication_codes)


tools = [normalize_medical_term, get_medication_diagnosis_proto]
print(f'Tools ready: {[t.name for t in tools]}')

## Step 5: System Prompt & Create Agent

The agent follows an iterative drill-down workflow using the `domainNarrowerByMedications` KG field:

1. **Extract** base diagnosis + medications from the clinical note
2. **Normalize** base diagnosis (domain=Problem) and each medication (domain=Medication)
3. **Iteratively call** `get_medication_diagnosis_proto` — pick the best candidate at each level, recurse until leaf
4. **Present** the inferred diagnosis with the full refinement path

In [ ]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are a diagnosis-inference agent. Given a clinical note that contains a BASE diagnosis (e.g., "tonsilitis") and a list of medications, you use the IMO Knowledge Graph to infer a MORE SPECIFIC diagnosis. The KG narrows the base diagnosis to only those child concepts that have ties to the given medications.

## WORKFLOW

### Step 1: Extract (NO TOOL CALLS)
From the clinical note, extract:
- The base diagnosis (a single problem term, e.g., "tonsilitis")
- The list of medications
Present a brief summary of what you extracted, then proceed to Step 2.

### Step 2: Normalize the Base Diagnosis
Call normalize_medical_term(input_term=<base diagnosis>, domain="Problem").
Capture its default_lexical_code as the diagnosis_code.

### Step 3: Normalize Each Medication
For each medication in the list, call normalize_medical_term(input_term=<med name>, domain="Medication").
Capture the default_lexical_code of each into a list of medication_codes. Skip any that fail to normalize.

### Step 4: Iteratively Drill Down to the Most Specific Concept

Call get_medication_diagnosis_proto(diagnosis_code=<current code>, medication_codes=[<med codes>]).
The response returns a list of narrower concepts, each with:
- `code` - its lexical code
- `title` - its display title
- `numberOfDomainChildren` - how many further narrower concepts exist below it

You must now iterate. On each round:

1. **Pick the best candidate** from the returned list using ONLY the clinical note's context (severity, laterality, chronicity, complications, comorbid conditions, etiology, functional class, EF, staging, etc. - whatever the note actually documents). Do not use medical knowledge to invent a match that is not supported by the note.
2. **Inspect `numberOfDomainChildren` on your pick**:
   - If it is `0` (or missing/null), you have reached a leaf. STOP iterating and go to Step 5 with this concept as the inferred diagnosis.
   - If it is `> 0`, this pick still has more specific children in the medication-linked hierarchy. Call get_medication_diagnosis_proto AGAIN with `diagnosis_code=<the pick's code>` and the SAME `medication_codes` list. Return to sub-step 1 with the new response.
3. **Bounds**: Iterate up to 8 rounds. If the tool returns an empty list at any level, back up and use the last non-empty concept as the final inferred diagnosis.
4. **Ambiguity**: If two candidates match the note equally well and both have children, prefer the one whose title better reflects the strongest clinical signal in the note (e.g., a documented EF, stage, or complication over a generic modifier). If two are equally supported and both are LEAVES, present both.

The tool is backed by the `domainNarrowerByMedications` KG field. If the tool returns an error, report it verbatim and stop - do NOT fabricate results.

### Step 5: Present the Inferred Diagnosis

#### Diagnosis Inference Using Medication

**Base Diagnosis:** (title) (`code`)
**Medications:** (list)

**Inferred Specific Diagnosis:** (final title) (`final code`)

**Refinement Path:**

| Depth | Code | Title | Children Below | Chosen? |
|---|---|---|---|---|
| 0 | (base code) | (base title) | - | starting concept |
| 1 | (code) | (title) | (n) | picked because ... |
| 2 | (code) | (title) | 0 | leaf - final |

For each row where you had multiple candidates, briefly note WHY you picked that candidate over the others, citing the specific fragment of the clinical note that supports the choice.

If the tool returned an error, state the error verbatim and note that this may indicate the KG service is unavailable or the codes are invalid.

## RULES
- Always normalize the base diagnosis and each medication first - never invent codes.
- ALWAYS pick from the candidates returned by the tool. Never fabricate a concept or invent a code.
- Recurse until `numberOfDomainChildren` is 0. Do not stop early just because a candidate "sounds specific enough" - the KG's own hierarchy is the source of truth for specificity.
- Never fabricate KG results. If the prototype endpoint returns an error, report it verbatim and stop.
- Do NOT use your own medical knowledge to narrow the diagnosis - only present concepts returned by the KG. You MAY use the note's own words to decide which returned candidate best fits.
- Before each tool call, briefly explain WHY you're calling it and which candidate you're drilling into."""

agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)
print('Diagnosis Inference Agent ready.')

## Step 6: Clinical Note

The following clinical note will be used to test the agent. It contains a base diagnosis
and medications that guide the inference through the KG hierarchy.

In [ ]:
CLINICAL_NOTE = """Chief Complaint: Sore throat and fever for 3 days.

HPI: 22-year-old female presents with sudden-onset severe sore throat, painful swallowing, and fever to 39.1°C beginning 3 days ago. No cough, no rhinorrhea, no hoarseness. Endorses malaise and mild headache. No prior similar episodes this year. Not immunocompromised, up to date on vaccinations. Recent exposure — roommate diagnosed with strep throat 5 days ago.

PMH: None. Non-smoker.

Medications (started at this visit):
- Amoxicillin 500 mg TID x 10 days

Allergies: NKDA.

Vitals: T 38.7°C, BP 118/72, HR 96, RR 16, SpO2 99%.

Exam: Symmetric tonsillar hypertrophy 3+ bilaterally with white exudate on both tonsils. Uvula midline (no deviation, no peritonsillar bulge or trismus). Tender bilateral anterior cervical lymphadenopathy. No rash. Lungs clear. No hepatosplenomegaly.

Centor Criteria: 4 (fever, tonsillar exudate, tender anterior cervical adenopathy, absence of cough).

Labs:
- Rapid strep antigen test: POSITIVE for group A Streptococcus
- Throat culture pending (confirmatory)
- CBC: WBC 14.2 with neutrophilic predominance
- Monospot: negative

Assessment:
1) Acute streptococcal tonsillitis (group A Streptococcus, GAS-positive rapid antigen), first documented episode — no history of recurrent tonsillitis. Not chronic. No peritonsillar abscess. Bacterial (not viral) etiology.
2) First-line penicillin family antibiotic prescribed given GAS.

Base Diagnosis:
- Tonsilitis

Medications:
- Amoxicillin

Please infer the more specific diagnosis using the medication list."""

print("Clinical note loaded (Tonsilitis + Amoxicillin from DS Agent).")
print(f"Length: {len(CLINICAL_NOTE)} characters")

## Step 7: Run the Agent

Send the clinical note to the agent and observe the iterative drill-down process.

In [ ]:
import asyncio
from IPython.display import display, HTML, Markdown
import html as html_module


class AgentUI:
    """Rich HTML display for agent streaming output."""

    @staticmethod
    def header():
        display(HTML("""
        <div style="text-align:center; padding:20px; background:linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    border-radius:12px; color:white; margin-bottom:16px;">
            <h2 style="margin:0;">Diagnosis Inference Agent (Medication-Guided)</h2>
            <p style="margin:4px 0 0 0; opacity:0.9;">Clinical Note &rarr; Normalize &rarr; domainNarrowerByMedications &rarr; Most Specific Diagnosis</p>
        </div>
        """))

    @staticmethod
    def tool_call(tool_name, parameters):
        params_json = html_module.escape(json.dumps(parameters, indent=2, default=str))
        display(HTML(f"""
        <div style="background:#e8f0fe; border:1px solid #1a73e8; border-radius:8px; padding:12px 16px; margin:8px 0; font-family:monospace; font-size:13px;">
            <b style="color:#1a73e8;">&#128295; Tool Call: {html_module.escape(tool_name)}</b>
            <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:8px; font-size:12px; overflow-x:auto;">{params_json}</pre>
        </div>
        """))

    @staticmethod
    def tool_result(tool_name, result):
        result_str = str(result)
        preview = html_module.escape(result_str[:400])
        full = html_module.escape(result_str)
        char_count = len(result_str)
        display(HTML(f"""
        <div style="background:#e6f4ea; border:1px solid #137333; border-radius:8px; padding:12px 16px; margin:8px 0; font-size:13px;">
            <b style="color:#137333;">&#9989; Result: {html_module.escape(tool_name)}</b>
            <details style="margin-top:8px;">
                <summary style="cursor:pointer; font-weight:600; font-size:12px; color:#137333;">View full result ({char_count:,} chars)</summary>
                <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:8px; font-size:11px; max-height:300px; overflow:auto;">{full}</pre>
            </details>
            <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:4px; font-size:11px; opacity:0.7;">{preview}{'...' if char_count > 400 else ''}</pre>
        </div>
        """))

    @staticmethod
    def agent_response(text):
        display(Markdown(text))

    @staticmethod
    def separator():
        display(HTML('<hr style="border:none; border-top:2px solid #dadce0; margin:16px 0;">'))


async def run_inference(clinical_note: str):
    """Run the diagnosis inference agent on a clinical note."""
    ui = AgentUI()
    ui.header()

    messages = [{'role': 'user', 'content': clinical_note}]
    final_content = ""

    async for chunk in agent.astream(
        {'messages': messages},
        stream_mode='updates'
    ):
        for node_name, node_output in chunk.items():
            if node_name == 'agent':
                msgs = node_output.get('messages', [])
                for msg in msgs:
                    if hasattr(msg, 'tool_calls') and msg.tool_calls:
                        for tc in msg.tool_calls:
                            ui.tool_call(tc['name'], tc.get('args', {}))
                    if hasattr(msg, 'content') and msg.content:
                        if isinstance(msg.content, str) and msg.content:
                            final_content = msg.content
                        elif isinstance(msg.content, list):
                            for block in msg.content:
                                if isinstance(block, dict) and block.get('type') == 'text':
                                    final_content += block['text']

            elif node_name == 'tools':
                msgs = node_output.get('messages', [])
                for msg in msgs:
                    if hasattr(msg, 'content'):
                        ui.tool_result(
                            getattr(msg, 'name', 'tool'),
                            msg.content
                        )

    if final_content:
        ui.separator()
        ui.agent_response(final_content)

    return final_content


result = asyncio.run(run_inference(CLINICAL_NOTE))

## Step 8: Interactive Chat Mode

Use this cell for multi-turn interaction with the agent.  
Paste any clinical note and the agent will infer the most specific diagnosis.

Commands: `quit` to end, `reset` to clear history.

In [ ]:
async def chat_interactive():
    """Multi-turn interactive agent chat."""
    messages = []
    ui = AgentUI()
    ui.header()

    while True:
        try:
            user_input = input('\nYou: ').strip()
        except (KeyboardInterrupt, EOFError):
            print('\nSession ended.')
            break

        if not user_input:
            continue
        if user_input.lower() in ('quit', 'stop', 'exit'):
            print('Session ended.')
            break
        if user_input.lower() == 'reset':
            messages = []
            print('Conversation reset. Paste a new clinical note.')
            continue

        display(HTML(f"""
        <div style="background:#f0f0f0; border-radius:8px; padding:10px 16px; margin:8px 0; font-size:14px;">
            <b>You:</b> {html_module.escape(user_input[:500])}{'...' if len(user_input) > 500 else ''}
        </div>
        """))

        messages.append({'role': 'user', 'content': user_input})

        final_content = ""
        try:
            async for chunk in agent.astream(
                {'messages': messages},
                stream_mode='updates'
            ):
                for node_name, node_output in chunk.items():
                    if node_name == 'agent':
                        msgs = node_output.get('messages', [])
                        for msg in msgs:
                            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                                for tc in msg.tool_calls:
                                    ui.tool_call(tc['name'], tc.get('args', {}))
                            if hasattr(msg, 'content') and msg.content:
                                if isinstance(msg.content, str) and msg.content:
                                    final_content = msg.content
                                elif isinstance(msg.content, list):
                                    for block in msg.content:
                                        if isinstance(block, dict) and block.get('type') == 'text':
                                            final_content += block['text']

                    elif node_name == 'tools':
                        msgs = node_output.get('messages', [])
                        for msg in msgs:
                            if hasattr(msg, 'content'):
                                ui.tool_result(
                                    getattr(msg, 'name', 'tool'),
                                    msg.content
                                )

        except KeyboardInterrupt:
            print('Generation interrupted.')

        if final_content:
            ui.separator()
            ui.agent_response(final_content)
            messages.append({'role': 'assistant', 'content': final_content})

        ui.separator()

await chat_interactive()

## Appendix: Additional Clinical Scenarios

Try them with the interactive chat:

| Scenario | Base Diagnosis | Key Medications | Clinical Signal |
|----------|---------------|-----------------|-----------------|
| T2DM + Metformin | Type 2 diabetes mellitus | Metformin | Poorly controlled, early CKD, neuropathy |
| Tonsilitis + Amoxicillin | Tonsillitis | Amoxicillin | Acute bacterial, GAS-positive, exudative |
| Pharyngitis + Penicillin V | Pharyngitis | Penicillin V | Acute streptococcal, GAS-positive |
| Otitis media + Amoxicillin | Otitis media | Amoxicillin, Ibuprofen | Acute suppurative, right ear, no perforation |
| Heart failure + GDMT | Heart failure | Sacubitril/valsartan, Carvedilol, Spironolactone, Dapagliflozin | HFrEF, LVEF 30%, ischemic etiology |